In [1]:
import sys
import json
import logging
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm import tqdm

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

In [2]:
BASE_DIR = Path.cwd().parent.parent
PROCESSED_DIR = BASE_DIR / "bigdata" / "processed"
ARTIFACTS_DIR = BASE_DIR / "artifacts" / "final_models"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
SEQ_LEN = 20
BATCH_SIZE = 128
EPOCHS = 50
PATIENCE = 5
VAL_START = '2022-01-01'
TEST_START = '2023-01-01'
TARGET_BINARY_COL = 'target_binary_20d'

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Устройство: {DEVICE}")

2026-05-25 00:02:54,921 [INFO] Устройство: cuda


In [13]:
df = pd.read_parquet(PROCESSED_DIR / "combined_features.parquet")
with open(PROCESSED_DIR / "feature_columns.txt") as f:
    all_feature_cols = [line.strip() for line in f if line.strip()]

df = df.sort_values(['date', 'ticker']).reset_index(drop=True)
logger.info(f"Загружено: {df.shape}, тикеров: {df['ticker'].nunique()}, признаков: {len(all_feature_cols)}")

MIN_TRAIN_DAYS = SEQ_LEN + 10   # 20 + 10 = 30 дней минимум до 2022-01-01

valid_tickers = []
for ticker in df['ticker'].unique():
    ticker_df = df[df['ticker'] == ticker]
    train_df = ticker_df[ticker_df['date'] < VAL_START]
    if len(train_df) >= MIN_TRAIN_DAYS:
        valid_tickers.append(ticker)

df = df[df['ticker'].isin(valid_tickers)].copy()
logger.info(f"Оставлено тикеров с историей >={MIN_TRAIN_DAYS} дней до {VAL_START}: {len(valid_tickers)}")

2026-05-25 00:13:41,392 [INFO] Загружено: (766539, 124), тикеров: 249, признаков: 109
2026-05-25 00:13:44,624 [INFO] Оставлено тикеров с историей >=30 дней до 2022-01-01: 233


In [14]:
# Отфильтруем данные для тренировки PCA (только train период)
train_pca_mask = df['date'] < VAL_START
X_train_pca_raw = df.loc[train_pca_mask, all_feature_cols].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train_pca_raw)

pca = PCA(n_components=0.95)   # сохраняем 95% дисперсии
pca.fit(X_scaled)

logger.info(f"PCA: {pca.n_components_} компонент, explained variance ratio: {pca.explained_variance_ratio_.sum():.4f}")

# Функция трансформации для любого набора строк
def transform_pca(df_subset):
    X = df_subset[all_feature_cols].ffill().values
    X_scaled = scaler.transform(X)
    return pca.transform(X_scaled)

# Применим ко всему датафрейму (по группам тикеров позже, чтобы не смешивать)
# Но мы будем трансформировать отдельно каждый тикер при создании последовательностей,
# используя общий scaler и pca.

2026-05-25 00:13:57,817 [INFO] PCA: 48 компонент, explained variance ratio: 0.9540


In [15]:
def create_pca_sequences(df, target_col, seq_len, test_start, val_start):
    X_train, y_train, X_val, y_val, X_test, y_test = [], [], [], [], [], []
    
    for ticker in df['ticker'].unique():
        ticker_data = df[df['ticker'] == ticker].sort_values('date').copy()
        if len(ticker_data) < seq_len + 10:
            continue
        
        # Заполняем пропуски в признаках (ffill) перед трансформацией
        ticker_data[all_feature_cols] = ticker_data[all_feature_cols].ffill()
        ticker_data = ticker_data.dropna(subset=all_feature_cols + [target_col])
        if len(ticker_data) < seq_len + 1:
            continue
        
        # Разделение
        train_df = ticker_data[ticker_data['date'] < val_start]
        val_df = ticker_data[(ticker_data['date'] >= val_start) & (ticker_data['date'] < test_start)]
        test_df = ticker_data[ticker_data['date'] >= test_start].copy()
        
        # Трансформация в PCA-пространство
        if len(train_df) >= seq_len + 1:
            train_pca = transform_pca(train_df)
            train_targets = train_df[target_col].values
            for i in range(len(train_pca) - seq_len):
                X_train.append(train_pca[i:i+seq_len])
                y_train.append(train_targets[i+seq_len])
        
        if len(val_df) >= seq_len + 1:
            val_pca = transform_pca(val_df)
            val_targets = val_df[target_col].values
            for i in range(len(val_pca) - seq_len):
                X_val.append(val_pca[i:i+seq_len])
                y_val.append(val_targets[i+seq_len])
        
        if len(test_df) >= seq_len + 1:
            test_pca = transform_pca(test_df)
            test_targets = test_df[target_col].values
            for i in range(len(test_pca) - seq_len):
                X_test.append(test_pca[i:i+seq_len])
                y_test.append(test_targets[i+seq_len])
    
    X_train = np.array(X_train, dtype=np.float32)
    y_train = np.array(y_train, dtype=np.float32)
    X_val = np.array(X_val, dtype=np.float32) if X_val else np.array([])
    y_val = np.array(y_val, dtype=np.float32) if y_val else np.array([])
    X_test = np.array(X_test, dtype=np.float32)
    y_test = np.array(y_test, dtype=np.float32)
    return X_train, y_train, X_val, y_val, X_test, y_test

X_train, y_train, X_val, y_val, X_test, y_test = create_pca_sequences(
    df, TARGET_BINARY_COL, SEQ_LEN, TEST_START, VAL_START
)
logger.info(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

2026-05-25 00:14:13,299 [INFO] Train: (560079, 20, 48), Val: (47870, 20, 48), Test: (91607, 20, 48)


In [16]:
class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=16, num_layers=1, dropout=0.5):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out, _ = self.gru(x)
        out = self.dropout(out[:, -1, :])
        return self.fc(out)

    def init_weights(self):
        for name, param in self.gru.named_parameters():
            if 'weight_ih' in name:
                torch.nn.init.xavier_uniform_(param)
            elif 'weight_hh' in name:
                torch.nn.init.orthogonal_(param)
            elif 'bias' in name:
                param.data.fill_(0.01)
        torch.nn.init.xavier_uniform_(self.fc.weight)
        if self.fc.bias is not None:
            self.fc.bias.data.fill_(0.01)

In [17]:
model = GRUModel(input_dim=pca.n_components_).to(DEVICE)
model.init_weights()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)

dataset = TensorDataset(torch.tensor(X_train).to(DEVICE), torch.tensor(y_train).to(DEVICE))
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

X_val_t = torch.tensor(X_val).to(DEVICE) if len(X_val) > 0 else None
y_val_t = torch.tensor(y_val).to(DEVICE) if len(y_val) > 0 else None

best_auc = -np.inf
counter = 0
best_state = None

In [18]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for batch_x, batch_y in tqdm(loader, desc=f"Epoch {epoch+1}"):
        optimizer.zero_grad()
        pred = model(batch_x).squeeze()
        loss = criterion(pred, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    
    if X_val_t is not None:
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t).squeeze()
            val_proba = torch.sigmoid(val_pred).cpu().numpy()
            val_auc = roc_auc_score(y_val, val_proba)
        logger.info(f"Epoch {epoch+1}: train_loss={avg_loss:.4f}, val_auc={val_auc:.4f}")
        
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = model.state_dict().copy()
            counter = 0
        else:
            counter += 1
            if counter >= PATIENCE:
                logger.info(f"Early stopping на эпохе {epoch+1}")
                break
        scheduler.step(1 - val_auc)
    else:
        logger.info(f"Epoch {epoch+1}: train_loss={avg_loss:.4f} (нет валидации)")

Epoch 1: 100%|██████████| 4376/4376 [00:16<00:00, 263.80it/s]
2026-05-25 00:14:30,807 [INFO] Epoch 1: train_loss=0.6837, val_auc=0.4796
Epoch 2: 100%|██████████| 4376/4376 [00:16<00:00, 271.75it/s]
2026-05-25 00:14:46,987 [INFO] Epoch 2: train_loss=0.6414, val_auc=0.4772
Epoch 3: 100%|██████████| 4376/4376 [00:17<00:00, 254.64it/s]
2026-05-25 00:15:04,258 [INFO] Epoch 3: train_loss=0.6302, val_auc=0.4823
Epoch 4: 100%|██████████| 4376/4376 [00:15<00:00, 274.05it/s]
2026-05-25 00:15:20,308 [INFO] Epoch 4: train_loss=0.6243, val_auc=0.5015
Epoch 5: 100%|██████████| 4376/4376 [00:17<00:00, 250.45it/s]
2026-05-25 00:15:37,863 [INFO] Epoch 5: train_loss=0.6200, val_auc=0.5141
Epoch 6: 100%|██████████| 4376/4376 [00:17<00:00, 256.24it/s]
2026-05-25 00:15:55,022 [INFO] Epoch 6: train_loss=0.6176, val_auc=0.5236
Epoch 7: 100%|██████████| 4376/4376 [00:16<00:00, 264.31it/s]
2026-05-25 00:16:11,659 [INFO] Epoch 7: train_loss=0.6154, val_auc=0.5080
Epoch 8: 100%|██████████| 4376/4376 [00:16<00:00

In [19]:
if best_state is not None:
    model.load_state_dict(best_state)
else:
    logger.warning("Лучшая модель не найдена, используем последнюю")

model.eval()
X_test_t = torch.tensor(X_test).to(DEVICE)
with torch.no_grad():
    y_pred_logits = model(X_test_t).squeeze().cpu().numpy()
y_pred_proba = 1.0 / (1.0 + np.exp(-y_pred_logits))
y_pred_class = (y_pred_proba >= 0.5).astype(int)

test_auc = roc_auc_score(y_test, y_pred_proba)
test_acc = accuracy_score(y_test, y_pred_class)
logger.info(f"Результаты на тесте: AUC = {test_auc:.4f}, Accuracy = {test_acc:.4f}")

# Сохраняем модель и трансформеры
if best_state is not None:
    torch.save(best_state, ARTIFACTS_DIR / "gru_pca_20d.pt")
    # Сохраняем scaler и pca для инференса
    import joblib
    joblib.dump(scaler, ARTIFACTS_DIR / "pca_scaler.pkl")
    joblib.dump(pca, ARTIFACTS_DIR / "pca_model.pkl")
    logger.info(f"Модель и PCA-трансформеры сохранены в {ARTIFACTS_DIR}")
else:
    logger.error("Модель не сохранена")

2026-05-25 00:19:34,199 [INFO] Результаты на тесте: AUC = 0.6004, Accuracy = 0.5301
2026-05-25 00:19:34,204 [INFO] Модель и PCA-трансформеры сохранены в d:\Storage\Projects\dpo\dpo\project\artifacts\final_models
